# Hacker News Parallel Distributed Computing Project (Dask)
University Level Project Notebook


In [ ]:
!pip install dask[complete] beautifulsoup4 requests pandas matplotlib

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import dask.bag as db
import time
import matplotlib.pyplot as plt
from collections import Counter


In [ ]:
STOP_WORDS = {
    'the','a','an','of','and','or','in','is','are','was','were','to','for',
    'on','at','by','with','from','it','its','as','be','this','that','we',
    'our','their','which','not','have','has','been','than','more','also',
    'they','he','she','us','can','will','all','any','but','up','out','had',
    'new','how','why','what','who','when','where','your','about','after',
    'into','over','some','such','would','there','if','do','get','just','now'
}


In [ ]:
def fetch_news(page):
    url = f"https://news.ycombinator.com/?p={page}"
    headers = {"User-Agent": "Mozilla/5.0"}
    
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    
    data = []
    rows = soup.select("tr.athing")
    
    for row in rows:
        title_tag = row.select_one(".titleline a")
        title = title_tag.text if title_tag else "N/A"
        
        subtext_row = row.find_next_sibling("tr")
        subtext = subtext_row.select_one(".subtext") if subtext_row else None
        
        score = 0
        comments = 0
        
        if subtext:
            score_tag = subtext.select_one(".score")
            if score_tag:
                score = int(score_tag.text.split()[0])
            
            links = subtext.find_all("a")
            if links:
                last = links[-1].text
                if "comment" in last:
                    try:
                        comments = int(last.split()[0])
                    except:
                        comments = 0
        
        data.append({
            'title': title,
            'score': score,
            'comments': comments,
            'page': page
        })
    
    return data


In [ ]:
test_data = fetch_news(1)
print(test_data[:5])


In [ ]:
pages = [1,2,3,4,5]

start = time.time()
seq_data = []
for p in pages:
    seq_data.extend(fetch_news(p))
seq_time = time.time() - start

print("Sequential Time:", seq_time)
print("Total posts:", len(seq_data))


In [ ]:
start = time.time()
bag = db.from_sequence(pages, npartitions=5)
results = bag.map(fetch_news).compute()
dask_data = [item for sublist in results for item in sublist]
dask_time = time.time() - start

print("Dask Time:", dask_time)
print("Total posts:", len(dask_data))


In [ ]:
import matplotlib.pyplot as plt
labels = ['Sequential','Dask']
times = [seq_time, dask_time]
plt.bar(labels, times)
plt.title("Sequential vs Dask Performance")
plt.ylabel("Time")
plt.show()


In [ ]:
df = pd.DataFrame(dask_data)
df.to_csv('hn_data.csv', index=False)
df.head()


In [ ]:
words = []
for t in df['title']:
    for w in str(t).lower().split():
        w = w.strip('.,()[]')
        if w not in STOP_WORDS and len(w) > 2:
            words.append(w)
freq = Counter(words)
top_words = freq.most_common(15)
top_words


In [ ]:
w = [x[0] for x in top_words]
c = [x[1] for x in top_words]
plt.figure(figsize=(12,6))
plt.bar(w,c)
plt.xticks(rotation=45)
plt.title('Top Words')
plt.show()
